Treniravimo ir testinių duomenų atsisiuntimas

In [4]:
classes = ["Car", "Bicycle", "Person"]
n = 1000


In [8]:
from openimages.download import download_dataset
import os

data_dir = "data"

if not os.path.exists(data_dir):
    os.makedirs(data_dir)

print(f"Downloading images for: {classes}")
try:
    download_dataset(data_dir, classes, limit=1000)
except TypeError:
    print("Download failed.")


2026-03-22  17:29:03 INFO Downloading 1000 train images for class 'car'
100%|██████████| 1000/1000 [00:50<00:00, 19.83it/s]
2026-03-22  17:29:53 INFO Downloading 1000 train images for class 'bicycle'
100%|██████████| 1000/1000 [00:51<00:00, 19.36it/s]
2026-03-22  17:30:46 INFO Downloading 1000 train images for class 'person'
100%|██████████| 1000/1000 [00:51<00:00, 19.56it/s]


Perskirsčiau nuotraukas rankiniu būdu taip, kad 800 nuotraukų būtų mokymuisi, 200 testavimui. Atskiros nuotraukos, data leakage neturėtų būti.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.backends.mps.is_available())
device = torch.device('mps')
print(device)

class ClassificationModel(nn.Module):
    def __init__(self):
        super(ClassificationModel, self).__init__()

        self.steps = nn.Sequential(
            # Blokas 1: 3 spalvų kanalų → 16 kanalų, 128×128 → 64×64
            nn.Conv2d(3, 16, kernel_size=3, padding=1), # konvoliucija
            nn.BatchNorm2d(16),     # mokymosi stabilizavimas, gera praktika pagal skaidres
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # naudoju max, nes man reikia distinktyvių rezultatų

            # Blokas 2: 16 → 32 kanalų, 64×64 → 32×32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Blokas 3: 32 → 64 kanalų, 32×32 → 16×16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3), # man reikia tik triju klasiu
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.steps(x)
        return x




True
mps


In [12]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transforms_train = transforms.Compose([
    transforms.Resize((128, 128)),       # pakeičia dydį į 128×128
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip()
])

transforms_test = transforms.Compose([
    transforms.Resize((128, 128)),       # pakeičia dydį į 128×128
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder('data/train', transform = transforms_train)
test_dataset = datasets.ImageFolder('data/test', transform = transforms_test)

num_workers = 2
batch_size = 25

train_loader = DataLoader(train_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = False)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')

Train: 2400, Test: 600
